# Sky Coefficient Prediction And Reconstruction

This notebook starts the new problem setup:
- Inputs: simultaneous SkyE and SkyW (or near/far) observations
- Intermediate target: science-location decomposition coefficients
- Final target: reconstructed science-location sky spectrum

It reuses robust FITS/decomposition I/O and reconstruction utilities from the generic model notebook, then builds a triplet dataset for coefficient transfer and spectrum reconstruction experiments.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from astropy.io import fits
from astropy.table import Table
from IPython.display import HTML, display

from sky_decomp.fit import reconstruct_component_spectra

# Reused constants
FACTOR = 1e14
PALACE_DIR = '../'

In [2]:
# Reused decomposition/context loading helpers
def _as_array(x):
    arr = np.asarray(x)
    if arr.dtype.kind in ('U', 'S', 'O'):
        return None
    return arr.astype(np.float32)


def _coerce_coef_hdu_to_table(coef_hdu):
    data = coef_hdu.data
    if isinstance(coef_hdu, (fits.BinTableHDU, fits.TableHDU)):
        return Table(data)

    arr = np.asarray(data, dtype=np.float32)
    if arr.ndim != 2:
        raise ValueError(f'Expected 2D COEF image, got shape={arr.shape}')

    n_coef = arr.shape[1]
    names = []
    for i in range(n_coef):
        key = f'COEF{i:04d}'
        names.append(str(coef_hdu.header.get(key, f'coef_{i:04d}')))
    return Table({name: arr[:, i] for i, name in enumerate(names)})


def _select_context_from_labels(meta, meta_upper, labels, base_name):
    e_key = f'SKYE_{base_name.upper()}'
    w_key = f'SKYW_{base_name.upper()}'
    if e_key not in meta_upper or w_key not in meta_upper:
        return None

    arr_e = _as_array(meta[meta_upper[e_key]])
    arr_w = _as_array(meta[meta_upper[w_key]])
    if arr_e is None or arr_w is None:
        raise ValueError(f'Labeled context columns for {base_name} are non-numeric.')

    is_e = labels == 'SKYE'
    is_w = labels == 'SKYW'
    if not np.all(is_e | is_w):
        bad = np.unique(labels[~(is_e | is_w)])
        raise ValueError(f'Unexpected label values: {bad}')

    return np.where(is_e, arr_e, arr_w).astype(np.float32)


def _table_to_float32_matrix(tbl, value_name):
    names = list(tbl.colnames)
    cols = []
    numeric_names = []
    for name in names:
        arr = _as_array(tbl[name])
        if arr is not None:
            cols.append(arr)
            numeric_names.append(name)

    if len(cols) == 0:
        raise ValueError(f'No numeric {value_name} columns found.')

    return np.column_stack(cols).astype(np.float32), numeric_names


def _build_context_matrix(meta, context_columns, kind):
    meta_upper = {c.upper(): c for c in meta.colnames}
    labels = None

    if kind in ('sky1', 'sky2'):
        label_col = 'SKY_NEAR_LABEL' if kind == 'sky1' else 'SKY_FAR_LABEL'
        if label_col not in meta_upper:
            raise KeyError(f'Missing required META label column: {label_col}')
        labels = np.char.upper(np.char.strip(np.asarray(meta[meta_upper[label_col]]).astype(str)))

    ctx_names = []
    ctx_cols = []
    missing_cols = []

    for cname in context_columns:
        key = cname.upper()

        # Science rows often use SCI_<name> naming; check this first for sci mode.
        if kind == 'sci':
            sci_key = f'SCI_{key}'
            if sci_key in meta_upper:
                arr = _as_array(meta[meta_upper[sci_key]])
                if arr is None:
                    raise ValueError(f'Context column {sci_key} is non-numeric.')
                ctx_names.append(cname)
                ctx_cols.append(arr)
                continue

        if key in meta_upper:
            arr = _as_array(meta[meta_upper[key]])
            if arr is None:
                raise ValueError(f'Context column {cname} is non-numeric.')
            ctx_names.append(cname)
            ctx_cols.append(arr)
            continue

        if labels is not None:
            arr = _select_context_from_labels(meta, meta_upper, labels, cname)
            if arr is not None:
                ctx_names.append(cname)
                ctx_cols.append(arr)
                continue

        missing_cols.append(cname)

    if missing_cols:
        raise KeyError(f'Missing requested context columns: {missing_cols}')
    if len(ctx_cols) == 0:
        raise ValueError('No usable context columns were assembled.')

    return np.column_stack(ctx_cols).astype(np.float32), ctx_names


def _find_chi2_column(meta_tbl):
    names = {c.upper(): c for c in meta_tbl.colnames}
    for cand in ('REDUCED_CHI2', 'CHI2_REDUCED', 'CHI2', 'RCHI2'):
        if cand in names:
            return names[cand]
    raise KeyError('No chi2-like column found in decomposition META table')


def read_decomp_dataset(decomp_fits_path, input_fits_path, context_columns, decomp_kind='sky1', return_chi2=False):
    if context_columns is None or len(context_columns) == 0:
        raise ValueError('context_columns must be a non-empty list.')

    kind = decomp_kind.lower()
    if kind not in ('sky1', 'sky2', 'sci'):
        raise ValueError("decomp_kind must be one of: 'sky1', 'sky2', 'sci'")

    with fits.open(decomp_fits_path) as hdul_dec, fits.open(input_fits_path) as hdul_in:
        coef_tbl = _coerce_coef_hdu_to_table(hdul_dec['COEF'])
        coef_mat, coef_names = _table_to_float32_matrix(coef_tbl, 'coefficient')

        meta = Table(hdul_in['META'].data)
        ctx_mat, ctx_names = _build_context_matrix(meta, context_columns, kind)

        if coef_mat.shape[0] != ctx_mat.shape[0]:
            raise ValueError(
                f'Row count mismatch: COEF has {coef_mat.shape[0]} rows, META has {ctx_mat.shape[0]} rows'
            )

        good = np.isfinite(coef_mat).all(axis=1) & np.isfinite(ctx_mat).all(axis=1)
        coef_mat = coef_mat[good]
        ctx_mat = ctx_mat[good]

        if not return_chi2:
            return coef_mat, ctx_mat, coef_names, ctx_names

        dec_meta = Table(hdul_dec['META'].data)
        chi2_col = _find_chi2_column(dec_meta)
        chi2_full = np.asarray(dec_meta[chi2_col], dtype=np.float64)
        chi2_used = chi2_full[good]

        if chi2_used.shape[0] != coef_mat.shape[0]:
            raise ValueError(
                f'Aligned chi2 rows ({chi2_used.shape[0]}) do not match coef rows ({coef_mat.shape[0]})'
            )

        return coef_mat, ctx_mat, coef_names, ctx_names, chi2_used

In [3]:
# New helper: build simultaneous triplets (near, far -> sci) from decomposition files
def build_triplet_coef_dataset(
    input_fits_path,
    sky_near_decomp_fits_path,
    sky_far_decomp_fits_path,
    sci_decomp_fits_path,
    context_columns,
    return_chi2=False,
):
    """Build aligned triplet arrays for coefficient-transfer experiments.

    Returns
    -------
    dict with keys:
      coef_near, coef_far, coef_sci
      ctx_near, ctx_far, ctx_sci
      coef_names, ctx_names, n_rows
      optional chi2_near, chi2_far, chi2_sci
    """
    near = read_decomp_dataset(
        decomp_fits_path=sky_near_decomp_fits_path,
        input_fits_path=input_fits_path,
        context_columns=context_columns,
        decomp_kind='sky1',
        return_chi2=return_chi2,
    )
    far = read_decomp_dataset(
        decomp_fits_path=sky_far_decomp_fits_path,
        input_fits_path=input_fits_path,
        context_columns=context_columns,
        decomp_kind='sky2',
        return_chi2=return_chi2,
    )
    sci = read_decomp_dataset(
        decomp_fits_path=sci_decomp_fits_path,
        input_fits_path=input_fits_path,
        context_columns=context_columns,
        decomp_kind='sci',
        return_chi2=return_chi2,
    )

    if return_chi2:
        coef_near, ctx_near, coef_names_n, ctx_names_n, chi2_near = near
        coef_far, ctx_far, coef_names_f, ctx_names_f, chi2_far = far
        coef_sci, ctx_sci, coef_names_s, ctx_names_s, chi2_sci = sci
    else:
        coef_near, ctx_near, coef_names_n, ctx_names_n = near
        coef_far, ctx_far, coef_names_f, ctx_names_f = far
        coef_sci, ctx_sci, coef_names_s, ctx_names_s = sci

    if coef_names_n != coef_names_f or coef_names_n != coef_names_s:
        raise ValueError('Coefficient name mismatch across near/far/sci decomposition products.')
    if ctx_names_n != ctx_names_f or ctx_names_n != ctx_names_s:
        raise ValueError('Context name mismatch across near/far/sci products.')

    n = min(coef_near.shape[0], coef_far.shape[0], coef_sci.shape[0])
    if n == 0:
        raise ValueError('No aligned rows available after filtering.')

    out = {
        'coef_near': coef_near[:n],
        'coef_far': coef_far[:n],
        'coef_sci': coef_sci[:n],
        'ctx_near': ctx_near[:n],
        'ctx_far': ctx_far[:n],
        'ctx_sci': ctx_sci[:n],
        'coef_names': coef_names_n,
        'ctx_names': ctx_names_n,
        'n_rows': n,
    }

    if return_chi2:
        out['chi2_near'] = chi2_near[:n]
        out['chi2_far'] = chi2_far[:n]
        out['chi2_sci'] = chi2_sci[:n]

    print(
        f"Triplet dataset built: n_rows={n}, n_coef={out['coef_near'].shape[1]}, n_ctx={out['ctx_near'].shape[1]}"
    )
    return out

In [4]:
# Reused reconstruction/prediction helpers for quick visual checks
def _meta_row_to_dict_upper(meta_row):
    names = list(meta_row.colnames) if hasattr(meta_row, 'colnames') else list(meta_row.dtype.names)
    return {str(k).upper(): k for k in names}


def _safe_float(x):
    arr = np.asarray(x)
    if arr.size == 0:
        raise ValueError('Empty value cannot be converted to float')
    if arr.shape != ():
        arr = arr.ravel()[0]
    return float(arr)


def _read_ext_row(hdul, extname, row_index):
    if extname not in [h.name for h in hdul]:
        raise KeyError(f'Missing required extension: {extname}')
    arr = np.asarray(hdul[extname].data, dtype=float)
    if arr.ndim == 1:
        return arr
    if arr.ndim >= 2:
        if row_index < 0 or row_index >= arr.shape[0]:
            raise IndexError(f'row_index {row_index} out of range [0, {arr.shape[0]-1}] for {extname}')
        return np.asarray(arr[row_index], dtype=float)
    raise ValueError(f'Unsupported ndim={arr.ndim} for extension {extname}')


def _display_scrollable_table(tbl):
    try:
        if hasattr(tbl, 'to_pandas'):
            df = tbl.to_pandas()
        else:
            df = pd.DataFrame(tbl)
    except Exception:
        print(tbl)
        return

    html = df.to_html(index=False)
    display(
        HTML(
            "<div style='max-width:100%; overflow-x:auto; border:1px solid #ddd; padding:6px;'>"
            + html
            + "</div>"
        )
    )


def _context_from_meta_row(meta_row, ctx_names_local, mode):
    umap = _meta_row_to_dict_upper(meta_row)
    mode = str(mode).lower()
    if mode not in ('near', 'far', 'sci'):
        raise ValueError(f'Unsupported mode: {mode}')

    label = None
    if mode == 'near' and 'SKY_NEAR_LABEL' in umap:
        label = str(meta_row[umap['SKY_NEAR_LABEL']]).strip().upper()
    elif mode == 'far' and 'SKY_FAR_LABEL' in umap:
        label = str(meta_row[umap['SKY_FAR_LABEL']]).strip().upper()

    out = []
    for cname in ctx_names_local:
        key = str(cname).upper()

        if mode == 'sci':
            sci_key = f'SCI_{key}'
            if sci_key in umap:
                out.append(_safe_float(meta_row[umap[sci_key]]))
                continue

        if key in umap:
            out.append(_safe_float(meta_row[umap[key]]))
            continue

        skye_key = f'SKYE_{key}'
        skyw_key = f'SKYW_{key}'
        has_skye = skye_key in umap
        has_skyw = skyw_key in umap

        if has_skye and has_skyw and mode in ('near', 'far'):
            v_e = _safe_float(meta_row[umap[skye_key]])
            v_w = _safe_float(meta_row[umap[skyw_key]])
            out.append(v_w if label == 'SKYW' else v_e)
            continue

        raise KeyError(f"Missing context field for '{cname}' in META row")

    return np.asarray(out, dtype=np.float32)


def _infer_base_dir_for_reconstruction():
    candidates = [Path.cwd().resolve(), Path.cwd().resolve().parent]
    if 'PALACE_DIR' in globals():
        try:
            p = Path(PALACE_DIR).resolve()
            candidates.extend([p, p.parent])
        except Exception:
            pass

    for cand in candidates:
        if (cand / 'palace' / 'PMD').exists() and (cand / 'Spectre_HR_LATMOS_Meftah_V1_350_1000nm.txt').exists():
            return cand

    raise FileNotFoundError('Could not infer reconstruction base_dir containing palace/PMD and solar reference file')


def predict_and_plot_three_fields(filename, row_index, predict_coef_from_context_fn, ctx_names_local):
    """Visual check helper.

    Parameters
    ----------
    predict_coef_from_context_fn : callable
        Function(ctx_row_phys) -> predicted coefficient vector.
    ctx_names_local : list[str]
        Context columns expected by the predictor.
    """
    path = Path(filename)
    if not path.exists():
        raise FileNotFoundError(f'File not found: {path}')

    with fits.open(path) as hdul:
        if 'META' not in [h.name for h in hdul]:
            raise KeyError('Missing META extension')
        meta = Table(hdul['META'].data)

        i = int(row_index)
        if i < 0 or i >= len(meta):
            raise IndexError(f'row_index {i} out of range [0, {len(meta)-1}]')

        meta_row = meta[i]
        print('META row used for context generation:')
        _display_scrollable_table(meta[i:i + 1])

        wave_local = _read_ext_row(hdul, 'WAVE', i)
        lsf_row = _read_ext_row(hdul, 'LSF_SCI', i)
        flux_near = _read_ext_row(hdul, 'FLUX_SKY_NEAR', i)
        flux_far = _read_ext_row(hdul, 'FLUX_SKY_FAR', i)
        flux_sci = _read_ext_row(hdul, 'FLUX_SCI', i)

    ctx_near = _context_from_meta_row(meta_row, ctx_names_local, mode='near')
    ctx_far = _context_from_meta_row(meta_row, ctx_names_local, mode='far')
    ctx_sci = _context_from_meta_row(meta_row, ctx_names_local, mode='sci')

    coef_near = predict_coef_from_context_fn(ctx_near)
    coef_far = predict_coef_from_context_fn(ctx_far)
    coef_sci = predict_coef_from_context_fn(ctx_sci)

    base_dir_guess = _infer_base_dir_for_reconstruction()

    comps_near = reconstruct_component_spectra(
        wave=wave_local,
        coef=coef_near,
        lsf_sigma=lsf_row / 2.35,
        n_spline_knots=25,
        base_dir=base_dir_guess,
    )
    comps_far = reconstruct_component_spectra(
        wave=wave_local,
        coef=coef_far,
        lsf_sigma=lsf_row / 2.35,
        n_spline_knots=25,
        base_dir=base_dir_guess,
    )
    comps_sci = reconstruct_component_spectra(
        wave=wave_local,
        coef=coef_sci,
        lsf_sigma=lsf_row / 2.35,
        n_spline_knots=25,
        base_dir=base_dir_guess,
    )

    fig = make_subplots(
        rows=3,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.04,
        subplot_titles=('Near Sky', 'Far Sky', 'Science Field'),
    )

    panel_data = [
        (1, flux_near, comps_near['total']),
        (2, flux_far, comps_far['total']),
        (3, flux_sci, comps_sci['total']),
    ]
    for row, flux_obs, flux_pred in panel_data:
        fig.add_trace(
            go.Scattergl(
                x=wave_local,
                y=flux_obs * FACTOR,
                mode='lines',
                name='observed' if row == 1 else None,
                showlegend=(row == 1),
                line=dict(color='#1f77b4', width=1.2),
            ),
            row=row,
            col=1,
        )
        fig.add_trace(
            go.Scattergl(
                x=wave_local,
                y=flux_pred,
                mode='lines',
                name='predicted reconstruction' if row == 1 else None,
                showlegend=(row == 1),
                line=dict(color='#d62728', width=1.2),
            ),
            row=row,
            col=1,
        )

    fig.update_yaxes(type='log', title_text='Flux', row=1, col=1)
    fig.update_yaxes(type='log', title_text='Flux', row=2, col=1)
    fig.update_yaxes(type='log', title_text='Flux', row=3, col=1)
    fig.update_xaxes(title_text='Wavelength [A]', row=3, col=1)
    fig.update_layout(
        template='plotly_white',
        height=980,
        title=f'Row {i}: predicted vs stored spectra (near/far/sci)',
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0.0),
    )
    fig.show()

    return {
        'file': str(path),
        'row_index': i,
        'wave': wave_local,
        'near': {'context': ctx_near, 'coef_pred': coef_near, 'flux_obs': flux_near, 'components': comps_near},
        'far': {'context': ctx_far, 'coef_pred': coef_far, 'flux_obs': flux_far, 'components': comps_far},
        'sci': {'context': ctx_sci, 'coef_pred': coef_sci, 'flux_obs': flux_sci, 'components': comps_sci},
    }

In [5]:
# Starter data load for coefficient prediction experiments
context_cols = [
    'alt',
    'moon_sep',
    'moon_alt',
    'sun_alt',
    'moon_illum',
    'airmass',
]

triplet = build_triplet_coef_dataset(
    input_fits_path='lvmsframe_median_stack_1.2.1_meta_only.fits',
    sky_near_decomp_fits_path='lvmsframe_median_stack_1.2.1_sky1_meta_coef.fits',
    sky_far_decomp_fits_path='lvmsframe_median_stack_1.2.1_sky2_meta_coef.fits',
    sci_decomp_fits_path='lvmsframe_median_stack_1.2.1_sci_meta_coef.fits',
    context_columns=context_cols,
    return_chi2=True,
)

print('Shapes:')
print('  coef_near', triplet['coef_near'].shape)
print('  coef_far ', triplet['coef_far'].shape)
print('  coef_sci ', triplet['coef_sci'].shape)
print('  ctx_near ', triplet['ctx_near'].shape)
print('  ctx_far  ', triplet['ctx_far'].shape)
print('  ctx_sci  ', triplet['ctx_sci'].shape)

Triplet dataset built: n_rows=17260, n_coef=442, n_ctx=6
Shapes:
  coef_near (17260, 442)
  coef_far  (17260, 442)
  coef_sci  (17260, 442)
  ctx_near  (17260, 6)
  ctx_far   (17260, 6)
  ctx_sci   (17260, 6)


## Filtering And Coefficient Learning

Apply the same robust row filtering used in the original notebook, then train a compact coefficient-prediction model that maps simultaneous near/far sky coefficients and geometry context to science-location coefficients.

In [17]:
# Filtering borrowed from the original notebook workflow, adapted to triplets
import plotly.express as px

def _coef_to_model_space(coef):
    return np.sqrt(np.clip(np.asarray(coef, dtype=np.float32), 0.0, None)).astype(np.float32)


def _coef_from_model_space(coef_model):
    coef_model = np.clip(np.asarray(coef_model, dtype=np.float32), 0.0, None)
    return np.square(coef_model).astype(np.float32)


def _kappa_sigma_row_mask(x, kappa=5.0, n_iter=3):
    x = np.asarray(x, dtype=np.float64)
    keep = np.isfinite(x).all(axis=1)
    if not np.any(keep):
        return keep
    for _ in range(n_iter):
        mu = np.nanmean(x[keep], axis=0)
        sig = np.nanstd(x[keep], axis=0)
        sig = np.where(np.isfinite(sig) & (sig > 0), sig, 1.0)
        within = np.all(np.abs(x - mu) <= (kappa * sig), axis=1)
        within &= np.isfinite(x).all(axis=1)
        new_keep = keep & within
        if new_keep.sum() == keep.sum() or new_keep.sum() == 0:
            break
        keep = new_keep
    return keep


def apply_triplet_filters(
    triplet_data,
    thin_every_n=2,
    chi2_qmax=90.0,
    chi2_min=0.0,
    chi2_max=10.0,
    hard_coef_bounds=None,
    kappa=6.0,
    kappa_iter=3,
):
    if hard_coef_bounds is None:
        hard_coef_bounds = {"feo": (0.0, 0.01), "atom_k": (0.0, 0.01)}

    coef_names_local = [str(n) for n in triplet_data["coef_names"]]
    coef_name_l = [n.lower() for n in coef_names_local]

    coef_near = np.asarray(triplet_data["coef_near"], dtype=np.float32)
    coef_far = np.asarray(triplet_data["coef_far"], dtype=np.float32)
    coef_sci = np.asarray(triplet_data["coef_sci"], dtype=np.float32)
    ctx_near = np.asarray(triplet_data["ctx_near"], dtype=np.float32)
    ctx_far = np.asarray(triplet_data["ctx_far"], dtype=np.float32)
    ctx_sci = np.asarray(triplet_data["ctx_sci"], dtype=np.float32)

    n0 = coef_near.shape[0]
    keep = np.ones(n0, dtype=bool)

    # Optional thinning, same as the original notebook.
    if int(thin_every_n) > 1:
        thin_mask = np.zeros(n0, dtype=bool)
        thin_mask[:: int(thin_every_n)] = True
        keep &= thin_mask
        print(f"Pre-chi2 thinning: every {int(thin_every_n)} row kept -> n_rows={thin_mask.sum()}")
    else:
        print(f"Pre-chi2 thinning disabled: n_rows={n0}")

    # Combined chi2 gating across near/far/sci when available.
    if all(k in triplet_data for k in ("chi2_near", "chi2_far", "chi2_sci")):
        chi2_stack = np.column_stack(
            [
                np.asarray(triplet_data["chi2_near"], dtype=np.float64),
                np.asarray(triplet_data["chi2_far"], dtype=np.float64),
                np.asarray(triplet_data["chi2_sci"], dtype=np.float64),
            ]
)
        chi2_combined = np.nanmax(chi2_stack, axis=1)
        chi2_finite = chi2_combined[np.isfinite(chi2_combined)]
        chi2_hi = np.nanpercentile(chi2_finite, chi2_qmax)
        chi2_upper = min(float(chi2_max), float(chi2_hi)) if chi2_max is not None else float(chi2_hi)
        chi2_mask = np.isfinite(chi2_combined) & (chi2_combined >= chi2_min) & (chi2_combined <= chi2_upper)
        keep &= chi2_mask
        print(
            f"Triplet chi2 filter: min={chi2_min:.3g}, qmax={chi2_qmax:.1f}%=>{chi2_hi:.3g}, "
            f"upper={chi2_upper:.3g} | keep={chi2_mask.sum()}/{len(chi2_mask)} ({100.0*chi2_mask.mean():.1f}%)"
        )
        fig_chi2_triplet = px.histogram(
            x=chi2_combined[keep],
            nbins=80,
            title="Triplet combined reduced chi2 distribution (rows used for training)",
            labels={"x": "max(reduced chi2 near/far/sci)", "y": "count"},
        )
        fig_chi2_triplet.update_layout(template="plotly_white", bargap=0.03)
        fig_chi2_triplet.show()

        # Context-parameter histograms for the same row subset shown in chi2 diagnostics.
        ctx_df = pd.DataFrame(ctx_sci[keep], columns=triplet_data["ctx_names"])
        ctx_long = ctx_df.melt(var_name="context_param", value_name="context_value")
        fig_ctx_hist = px.histogram(
            ctx_long,
            x="context_value",
            facet_col="context_param",
            facet_col_wrap=min(3, max(1, len(triplet_data["ctx_names"]))),
            nbins=70,
            title="Context parameter distributions (rows used after chi2 filter)",
            labels={"context_value": "value", "count": "count", "context_param": "context"},
        )
        # Force independent histogram binning in each subplot.
        for i, tr in enumerate(fig_ctx_hist.data):
            tr.update(bingroup=f"ctx_{i}", autobinx=True, xbins=dict())
        fig_ctx_hist.for_each_annotation(lambda a: a.update(text=a.text.replace("context_param=", "")))
        fig_ctx_hist.update_xaxes(matches=None)
        fig_ctx_hist.update_yaxes(matches=None)
        fig_ctx_hist.update_layout(template="plotly_white", bargap=0.03, height=650)
        fig_ctx_hist.show()
    else:
        print("Triplet chi2 columns not present; chi2 filtering skipped.")

    # Manual hard coefficient bounds applied to all three fields.
    for cname, (lo, hi) in hard_coef_bounds.items():
        idxs = np.where(np.array(coef_name_l) == str(cname).lower())[0]
        if idxs.size == 0:
            print(f"Manual hard clip: coefficient {cname} not found; skipping.")
            continue
        j = int(idxs[0])
        within = (
            np.isfinite(coef_near[:, j]) & (coef_near[:, j] >= lo) & (coef_near[:, j] <= hi)
            & np.isfinite(coef_far[:, j]) & (coef_far[:, j] >= lo) & (coef_far[:, j] <= hi)
            & np.isfinite(coef_sci[:, j]) & (coef_sci[:, j] >= lo) & (coef_sci[:, j] <= hi)
)
        keep &= within
        print(
            f"Manual hard clip {cname}: [{lo:.3g}, {hi:.3g}] | kept {within.sum()}/{within.size} ({100.0 * within.mean():.1f}%)"
        )

    # Kappa-sigma clipping in concatenated coefficient space.
    coef_concat = np.hstack([coef_near, coef_far, coef_sci]).astype(np.float32)
    kappa_mask = _kappa_sigma_row_mask(coef_concat, kappa=float(kappa), n_iter=int(kappa_iter))
    keep &= kappa_mask
    print(
        f"Kappa-sigma filter (kappa={kappa:.1f}): kept {kappa_mask.sum()}/{len(kappa_mask)} ({100.0 * kappa_mask.mean():.1f}%)"
    )

    if keep.sum() == 0:
        raise RuntimeError("Filtering removed all rows; relax thresholds.")

    out = {
        "coef_near": coef_near[keep],
        "coef_far": coef_far[keep],
        "coef_sci": coef_sci[keep],
        "ctx_near": ctx_near[keep],
        "ctx_far": ctx_far[keep],
        "ctx_sci": ctx_sci[keep],
        "coef_names": coef_names_local,
        "ctx_names": list(triplet_data["ctx_names"]),
        "mask": keep,
    }
    for k in ("chi2_near", "chi2_far", "chi2_sci"):
        if k in triplet_data:
            out[k] = np.asarray(triplet_data[k])[keep]

    print(
        f"Filtered triplet shapes: near={out['coef_near'].shape}, far={out['coef_far'].shape}, "
        f"sci={out['coef_sci'].shape}"
    )
    return out


filtered_triplet = apply_triplet_filters(
    triplet,
    thin_every_n=2,
    chi2_qmax=90.0,
    chi2_min=0.0,
    chi2_max=10.0,
    hard_coef_bounds={"feo": (0.0, 0.01), "atom_k": (0.0, 0.01)},
    kappa=6.0,
    kappa_iter=3,
)

Pre-chi2 thinning: every 2 row kept -> n_rows=8630
Triplet chi2 filter: min=0, qmax=90.0%=>5.7, upper=5.7 | keep=15534/17260 (90.0%)


Manual hard clip feo: [0, 0.01] | kept 11752/17260 (68.1%)
Manual hard clip atom_k: [0, 0.01] | kept 12672/17260 (73.4%)
Kappa-sigma filter (kappa=6.0): kept 15641/17260 (90.6%)
Filtered triplet shapes: near=(4231, 442), far=(4231, 442), sci=(4231, 442)


In [7]:
# Shared ML utilities for coefficient prediction models
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset


class RobustScaler:
    def fit(self, x):
        x = np.asarray(x, dtype=np.float32)
        self.med_ = np.nanmedian(x, axis=0)
        q25 = np.nanpercentile(x, 25, axis=0)
        q75 = np.nanpercentile(x, 75, axis=0)
        iqr = q75 - q25
        self.scale_ = np.where(iqr > 1e-8, iqr, 1.0).astype(np.float32)
        return self

    def transform(self, x):
        x = np.asarray(x, dtype=np.float32)
        return (x - self.med_) / self.scale_

    def inverse_transform(self, x):
        x = np.asarray(x, dtype=np.float32)
        return x * self.scale_ + self.med_


def _set_reproducibility(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def split_indices(n, train_frac=0.8, val_frac=0.1, seed=42):
    rng = np.random.default_rng(seed)
    idx = np.arange(n)
    rng.shuffle(idx)
    n_train = int(train_frac * n)
    n_val = int(val_frac * n)
    train_idx = idx[:n_train]
    val_idx = idx[n_train:n_train + n_val]
    test_idx = idx[n_train + n_val:]
    return train_idx, val_idx, test_idx


def _make_loader(*arrays, batch_size=256, shuffle=False):
    tensors = [torch.from_numpy(np.asarray(a, dtype=np.float32)) for a in arrays]
    ds = TensorDataset(*tensors)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, drop_last=False)

## Coefficient Prediction Model

This section trains the geometry-aware model that predicts science decomposition coefficients from near/far coefficients and context. These predicted coefficients are later passed to spectral reconstruction.

In [12]:
# Geometry-aware coefficient prediction model
class SkyCoeffGeoWeightedNet(nn.Module):
    """Predict science decomposition coefficients from near/far coefficients and geometry.

    Model design
    ------------
    Inputs are already normalized in model space:
      - near_n: normalized near-field coefficients, shape [batch, n_coef]
      - far_n: normalized far-field coefficients, shape [batch, n_coef]
      - delta_ctx_n: science context offset from near/far midpoint, shape [batch, n_ctx]
      - sci_ctx_n: normalized absolute science context, shape [batch, n_ctx]

    The network performs direct regression (not residual-on-average):
      concatenated features -> MLP -> predicted science coefficients.
    """
    def __init__(self, n_coef, n_ctx, hidden=256):
        super().__init__()
        self.n_coef = int(n_coef)
        geom_dim = 2 * n_ctx  # [delta_ctx_n, sci_ctx_n]
        # Direct predictor: map (near, far, geometry) -> science coefficients.
        self.predictor_net = nn.Sequential(
            nn.Linear(2 * n_coef + geom_dim, hidden),
            nn.GELU(),
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Linear(hidden, n_coef),
        )

    def forward(self, near_n, far_n, delta_ctx_n, sci_ctx_n):
        # Build a single feature vector per row.
        geom = torch.cat([delta_ctx_n, sci_ctx_n], dim=-1)
        features = torch.cat([near_n, far_n, geom], dim=-1)
        return self.predictor_net(features)


def train_coeff_prediction_model_geo_weighted(filtered, n_epochs=120, batch_size=256, lr=1e-3, seed=42):
    _set_reproducibility(seed)

    coef_near = np.asarray(filtered["coef_near"], dtype=np.float32)
    coef_far = np.asarray(filtered["coef_far"], dtype=np.float32)
    coef_sci = np.asarray(filtered["coef_sci"], dtype=np.float32)
    ctx_near = np.asarray(filtered["ctx_near"], dtype=np.float32)
    ctx_far = np.asarray(filtered["ctx_far"], dtype=np.float32)
    ctx_sci = np.asarray(filtered["ctx_sci"], dtype=np.float32)

    n = coef_near.shape[0]
    train_idx, val_idx, test_idx = split_indices(n, seed=seed)

    # Train in transformed coefficient space for better stability and dynamic range.
    near_model = _coef_to_model_space(coef_near)
    far_model = _coef_to_model_space(coef_far)
    sci_model = _coef_to_model_space(coef_sci)

    coef_scaler = RobustScaler().fit(np.vstack([near_model[train_idx], far_model[train_idx], sci_model[train_idx]]))
    ctx_scaler = RobustScaler().fit(np.vstack([ctx_near[train_idx], ctx_far[train_idx], ctx_sci[train_idx]]))

    near_n = np.clip(coef_scaler.transform(near_model), -25.0, 25.0)
    far_n = np.clip(coef_scaler.transform(far_model), -25.0, 25.0)
    sci_n = np.clip(coef_scaler.transform(sci_model), -25.0, 25.0)

    near_ctx_n = np.clip(ctx_scaler.transform(ctx_near), -25.0, 25.0)
    far_ctx_n = np.clip(ctx_scaler.transform(ctx_far), -25.0, 25.0)
    sci_ctx_n = np.clip(ctx_scaler.transform(ctx_sci), -25.0, 25.0)
    # Geometry feature: science context offset relative to near/far midpoint.
    # This is only an input descriptor, not a prediction baseline.
    delta_ctx_n = sci_ctx_n - 0.5 * (near_ctx_n + far_ctx_n)

    tr_loader = _make_loader(
        near_n[train_idx], far_n[train_idx], delta_ctx_n[train_idx], sci_ctx_n[train_idx], sci_n[train_idx],
        batch_size=batch_size,
        shuffle=True,
    )
    va_loader = _make_loader(
        near_n[val_idx], far_n[val_idx], delta_ctx_n[val_idx], sci_ctx_n[val_idx], sci_n[val_idx],
        batch_size=batch_size,
        shuffle=False,
    )

    if torch.cuda.is_available():
        device = "cuda"
    elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available() and batch_size > 256:
        device = "mps"
    else:
        device = "cpu"

    model = SkyCoeffGeoWeightedNet(n_coef=near_n.shape[1], n_ctx=sci_ctx_n.shape[1], hidden=256).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)

    best = {"val_loss": np.inf, "state": None, "epoch": -1}
    history = []

    for epoch in range(1, n_epochs + 1):
        model.train()
        tr_loss = 0.0
        tr_batches = 0
        for near_b, far_b, dctx_b, scictx_b, target_b in tr_loader:
            near_b = near_b.to(device)
            far_b = far_b.to(device)
            dctx_b = dctx_b.to(device)
            scictx_b = scictx_b.to(device)
            target_b = target_b.to(device)

            opt.zero_grad(set_to_none=True)
            pred_b = model(near_b, far_b, dctx_b, scictx_b)
            loss = F.smooth_l1_loss(pred_b, target_b)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            tr_loss += float(loss.item())
            tr_batches += 1

        model.eval()
        va_loss = 0.0
        va_batches = 0
        with torch.no_grad():
            for near_b, far_b, dctx_b, scictx_b, target_b in va_loader:
                near_b = near_b.to(device)
                far_b = far_b.to(device)
                dctx_b = dctx_b.to(device)
                scictx_b = scictx_b.to(device)
                target_b = target_b.to(device)
                pred_b = model(near_b, far_b, dctx_b, scictx_b)
                loss = F.smooth_l1_loss(pred_b, target_b)
                va_loss += float(loss.item())
                va_batches += 1

        row = {
            "epoch": epoch,
            "train_loss": tr_loss / max(tr_batches, 1),
            "val_loss": va_loss / max(va_batches, 1),
        }
        history.append(row)

        if row["val_loss"] < best["val_loss"]:
            best["val_loss"] = row["val_loss"]
            best["state"] = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best["epoch"] = epoch

        if epoch == 1 or epoch % 10 == 0:
            print(f"[geo] epoch={epoch:03d} train={row['train_loss']:.5f} val={row['val_loss']:.5f}")

    if best["state"] is not None:
        model.load_state_dict(best["state"])

    coeff_model_artifacts_geo = {
        "model": model,
        "device": device,
        "coef_scaler": coef_scaler,
        "ctx_scaler": ctx_scaler,
        "history": history,
        "best_epoch": best["epoch"],
        "best_val_loss": best["val_loss"],
        "train_idx": train_idx,
        "val_idx": val_idx,
        "test_idx": test_idx,
        "coef_names": list(filtered["coef_names"]),
        "ctx_names": list(filtered["ctx_names"]),
    }
    return coeff_model_artifacts_geo


def predict_sci_coefficients_geo_weighted(
    coeff_model_artifacts_geo,
    coef_near_phys,
    coef_far_phys,
    ctx_near_phys,
    ctx_far_phys,
    ctx_sci_phys,
):
    """Run inference for science coefficients in physical coefficient units.

    This function mirrors training-time preprocessing exactly:
    physical coef -> model-space transform -> robust scaling -> network -> inverse scaling -> physical coef.
    """
    model = coeff_model_artifacts_geo["model"]
    device = coeff_model_artifacts_geo["device"]
    coef_scaler = coeff_model_artifacts_geo["coef_scaler"]
    ctx_scaler = coeff_model_artifacts_geo["ctx_scaler"]

    coef_near_phys = np.asarray(coef_near_phys, dtype=np.float32)
    coef_far_phys = np.asarray(coef_far_phys, dtype=np.float32)
    ctx_near_phys = np.asarray(ctx_near_phys, dtype=np.float32)
    ctx_far_phys = np.asarray(ctx_far_phys, dtype=np.float32)
    ctx_sci_phys = np.asarray(ctx_sci_phys, dtype=np.float32)

    if coef_near_phys.ndim == 1:
        coef_near_phys = coef_near_phys[None, :]
    if coef_far_phys.ndim == 1:
        coef_far_phys = coef_far_phys[None, :]
    if ctx_near_phys.ndim == 1:
        ctx_near_phys = ctx_near_phys[None, :]
    if ctx_far_phys.ndim == 1:
        ctx_far_phys = ctx_far_phys[None, :]
    if ctx_sci_phys.ndim == 1:
        ctx_sci_phys = ctx_sci_phys[None, :]

    near_n = np.clip(coef_scaler.transform(_coef_to_model_space(coef_near_phys)), -25.0, 25.0).astype(np.float32)
    far_n = np.clip(coef_scaler.transform(_coef_to_model_space(coef_far_phys)), -25.0, 25.0).astype(np.float32)

    near_ctx_n = np.clip(ctx_scaler.transform(ctx_near_phys), -25.0, 25.0).astype(np.float32)
    far_ctx_n = np.clip(ctx_scaler.transform(ctx_far_phys), -25.0, 25.0).astype(np.float32)
    sci_ctx_n = np.clip(ctx_scaler.transform(ctx_sci_phys), -25.0, 25.0).astype(np.float32)
    # Same geometry descriptor used at training time.
    delta_ctx_n = sci_ctx_n - 0.5 * (near_ctx_n + far_ctx_n)

    with torch.no_grad():
        pred_n = model(
            torch.from_numpy(near_n).to(device),
            torch.from_numpy(far_n).to(device),
            torch.from_numpy(delta_ctx_n).to(device),
            torch.from_numpy(sci_ctx_n).to(device),
        )
        pred_n = pred_n.cpu().numpy()

    pred_model = coef_scaler.inverse_transform(pred_n)
    pred_phys = _coef_from_model_space(pred_model)
    return pred_phys

In [9]:
# Train and evaluate coefficient-prediction model
coeff_model_artifacts = train_coeff_prediction_model_geo_weighted(
    filtered_triplet,
    n_epochs=120,
    batch_size=256,
    lr=1e-3,
    seed=42,
)
print(f"Best epoch: {coeff_model_artifacts['best_epoch']} | best val loss: {coeff_model_artifacts['best_val_loss']:.6f}")

test_idx_coeff = np.asarray(coeff_model_artifacts["test_idx"], dtype=int)
coef_pred_test = predict_sci_coefficients_geo_weighted(
    coeff_model_artifacts,
    coef_near_phys=filtered_triplet["coef_near"][test_idx_coeff],
    coef_far_phys=filtered_triplet["coef_far"][test_idx_coeff],
    ctx_near_phys=filtered_triplet["ctx_near"][test_idx_coeff],
    ctx_far_phys=filtered_triplet["ctx_far"][test_idx_coeff],
    ctx_sci_phys=filtered_triplet["ctx_sci"][test_idx_coeff],
)
coef_true_test = np.asarray(filtered_triplet["coef_sci"][test_idx_coeff], dtype=np.float32)

rmse = np.sqrt(np.mean((coef_pred_test - coef_true_test) ** 2, axis=0))
mae = np.mean(np.abs(coef_pred_test - coef_true_test), axis=0)

corr = []
for j in range(coef_true_test.shape[1]):
    x = coef_true_test[:, j]
    y = coef_pred_test[:, j]
    if np.std(x) < 1e-12 or np.std(y) < 1e-12:
        corr.append(np.nan)
    else:
        corr.append(float(np.corrcoef(x, y)[0, 1]))
corr = np.asarray(corr)

print("\nCoefficient prediction test summary:")
print(f"  mean RMSE   = {np.nanmean(rmse):.6g}")
print(f"  median RMSE = {np.nanmedian(rmse):.6g}")
print(f"  mean MAE    = {np.nanmean(mae):.6g}")
print(f"  mean corr   = {np.nanmean(corr):.6g}")
print(f"  median corr = {np.nanmedian(corr):.6g}")

coef_names_local = np.asarray(filtered_triplet["coef_names"])
worst_idx = np.argsort(rmse)[-10:][::-1]
print("\nWorst 10 coefficients by RMSE:")
for k in worst_idx:
    print(f"  {coef_names_local[k]}: rmse={rmse[k]:.6g}, mae={mae[k]:.6g}, corr={corr[k]:.4f}")

[geo] epoch=001 train=0.35267 val=0.29893
[geo] epoch=010 train=0.18143 val=0.20951
[geo] epoch=020 train=0.13435 val=0.20091
[geo] epoch=030 train=0.10997 val=0.18842
[geo] epoch=040 train=0.09501 val=0.19177
[geo] epoch=050 train=0.08301 val=0.18789
[geo] epoch=060 train=0.07742 val=0.18692
[geo] epoch=070 train=0.07226 val=0.18876
[geo] epoch=080 train=0.06756 val=0.18988
[geo] epoch=090 train=0.06200 val=0.18695
[geo] epoch=100 train=0.06032 val=0.18840
[geo] epoch=110 train=0.05736 val=0.18737
[geo] epoch=120 train=0.05433 val=0.18662
Best epoch: 48 | best val loss: 0.185891

Coefficient prediction test summary:
  mean RMSE   = 48.7346
  median RMSE = 0.00731605
  mean MAE    = 35.9439
  mean corr   = 0.756243
  median corr = 0.926979

Worst 10 coefficients by RMSE:
  OH_122: rmse=15177.4, mae=11669.4, corr=0.9843
  OH_037: rmse=3588.26, mae=2619.95, corr=0.9818
  OH_046: rmse=537.184, mae=393.488, corr=0.9793
  OH_036: rmse=453.008, mae=54.4614, corr=0.8317
  OH_038: rmse=315.948

## Conditional VAE Coefficient Model (Comparison)

This section trains a conditional VAE (cVAE) to model $p(\mathrm{coef}_{\mathrm{sci}} \mid \mathrm{coef}_{\mathrm{near}}, \mathrm{coef}_{\mathrm{far}}, \mathrm{context})$ and compares it against the current direct regressor.

Why this can help:
1. It can represent one-to-many mappings via latent variables.
2. It provides both deterministic prior-mean predictions and stochastic sampled predictions.
3. It lets us test whether multimodal structure improves coefficient prediction quality.

In [28]:
# cVAE model and helpers for conditional coefficient prediction

def _build_physics_group_ids(coef_names):
    """Map coefficients to coarse physics groups for embedding-aware cVAE encoding/decoding."""
    group_defs = [
        ("moon_bs", ("moon_bs",)),
        ("oh", ("oh", "oh_")),
        ("atom", ("atom",)),
        ("o2", ("o2",)),
        ("feo", ("feo",)),
        ("ho2", ("ho2",)),
        ("diffuse", ("diffuse",)),
        ("continuum", ("continuum",)),
    ]
    group_names = [g[0] for g in group_defs] + ["other"]
    out = np.full(len(coef_names), len(group_names) - 1, dtype=np.int64)
    for i, n in enumerate(coef_names):
        lname = str(n).lower()
        for j, (_, prefixes) in enumerate(group_defs):
            if any(lname.startswith(p) or (p in lname and p in ("diffuse", "continuum")) for p in prefixes):
                out[i] = j
                break
    return out, group_names


class ConditionalCoeffVAE(nn.Module):
    """Conditional VAE for science coefficients given near/far coefficients and context."""

    def __init__(
        self,
        n_coef,
        n_ctx,
        z_dim=8,
        hidden=256,
        coef_group_ids=None,
        n_groups=1,
        group_emb_dim=8,
        spline_indices=None,
        spline_rank=4,
        oh_indices=None,
        oh_rank=8,
        continuum_indices=None,
        continuum_rank=8,
    ):
        super().__init__()
        self.n_coef = int(n_coef)
        self.n_ctx = int(n_ctx)
        self.z_dim = int(z_dim)
        cond_dim = 2 * self.n_coef + 2 * self.n_ctx

        if coef_group_ids is None:
            coef_group_ids = torch.zeros(self.n_coef, dtype=torch.long)
        if coef_group_ids.numel() != self.n_coef:
            raise ValueError(f"coef_group_ids length mismatch: expected {self.n_coef}, got {coef_group_ids.numel()}")
        self.register_buffer("coef_group_ids", coef_group_ids.long())

        self.group_emb = nn.Embedding(int(max(1, n_groups)), int(max(1, group_emb_dim)))
        self.enc_group_scale = nn.Linear(int(max(1, group_emb_dim)), 1)
        self.enc_group_bias = nn.Linear(int(max(1, group_emb_dim)), 1)
        self.dec_group_bias = nn.Linear(int(max(1, group_emb_dim)), 1)

        self.encoder = nn.Sequential(
            nn.Linear(cond_dim + self.n_coef, hidden),
            nn.GELU(),
            nn.Linear(hidden, hidden),
            nn.GELU(),
        )
        self.enc_mu = nn.Linear(hidden, self.z_dim)
        self.enc_logvar = nn.Linear(hidden, self.z_dim)

        self.prior_net = nn.Sequential(
            nn.Linear(cond_dim, hidden),
            nn.GELU(),
            nn.Linear(hidden, hidden),
            nn.GELU(),
        )
        self.prior_mu = nn.Linear(hidden, self.z_dim)
        self.prior_logvar = nn.Linear(hidden, self.z_dim)

        self.decoder_backbone = nn.Sequential(
            nn.Linear(cond_dim + self.z_dim, hidden),
            nn.GELU(),
            nn.Linear(hidden, hidden),
            nn.GELU(),
        )
        self.decoder_out = nn.Linear(hidden, self.n_coef)

        spline_indices = np.asarray(spline_indices if spline_indices is not None else [], dtype=np.int64)
        oh_indices = np.asarray(oh_indices if oh_indices is not None else [], dtype=np.int64)
        continuum_indices = np.asarray(continuum_indices if continuum_indices is not None else [], dtype=np.int64)

        self.register_buffer("spline_idx", torch.from_numpy(spline_indices.astype(np.int64)), persistent=False)
        self.register_buffer("oh_idx", torch.from_numpy(oh_indices.astype(np.int64)), persistent=False)
        self.register_buffer("continuum_idx", torch.from_numpy(continuum_indices.astype(np.int64)), persistent=False)

        self.spline_rank = int(max(1, spline_rank))
        self.has_spline_head = int(spline_indices.size) >= 3
        if self.has_spline_head:
            n_spline = int(spline_indices.size)
            self.spline_factor = nn.Linear(hidden, self.spline_rank)
            self.spline_basis = nn.Parameter(0.01 * torch.randn(self.spline_rank, n_spline))

        self.oh_rank = int(max(1, oh_rank))
        self.has_oh_head = int(oh_indices.size) >= 3
        if self.has_oh_head:
            n_oh = int(oh_indices.size)
            self.oh_factor = nn.Linear(hidden, self.oh_rank)
            self.oh_basis = nn.Parameter(0.01 * torch.randn(self.oh_rank, n_oh))

        self.continuum_rank = int(max(1, continuum_rank))
        self.has_continuum_head = int(continuum_indices.size) >= 2
        if self.has_continuum_head:
            n_cont = int(continuum_indices.size)
            self.ctx_continuum_backbone = nn.Sequential(
                nn.Linear(2 * self.n_ctx, hidden // 2),
                nn.GELU(),
                nn.Linear(hidden // 2, self.continuum_rank),
            )
            self.continuum_basis = nn.Parameter(0.01 * torch.randn(self.continuum_rank, n_cont))

    def _group_embedding(self):
        return self.group_emb(self.coef_group_ids)

    def _augment_coef_for_encode(self, coef):
        emb = self._group_embedding()
        g_scale = self.enc_group_scale(emb).view(1, -1)
        g_bias = self.enc_group_bias(emb).view(1, -1)
        return coef * (1.0 + 0.10 * torch.tanh(g_scale)) + 0.10 * torch.tanh(g_bias)

    def _build_cond(self, near_n, far_n, delta_ctx_n, sci_ctx_n):
        geom = torch.cat([delta_ctx_n, sci_ctx_n], dim=-1)
        return torch.cat([near_n, far_n, geom], dim=-1)

    def encode(self, near_n, far_n, delta_ctx_n, sci_ctx_n, sci_n):
        cond = self._build_cond(near_n, far_n, delta_ctx_n, sci_ctx_n)
        sci_aug = self._augment_coef_for_encode(sci_n)
        h = self.encoder(torch.cat([cond, sci_aug], dim=-1))
        return self.enc_mu(h), self.enc_logvar(h), cond

    def prior(self, cond):
        h = self.prior_net(cond)
        return self.prior_mu(h), self.prior_logvar(h)

    def decode(self, cond, z):
        h = self.decoder_backbone(torch.cat([cond, z], dim=-1))
        coef_hat = self.decoder_out(h)

        emb = self._group_embedding()
        group_bias = 0.05 * torch.tanh(self.dec_group_bias(emb)).view(1, -1)
        coef_hat = coef_hat + group_bias

        # ND-style low-rank heads for structured coefficient families.
        if self.has_spline_head:
            spline_resid = self.spline_factor(h) @ self.spline_basis
            coef_hat[:, self.spline_idx] = coef_hat[:, self.spline_idx] + spline_resid

        if self.has_oh_head:
            oh_resid = self.oh_factor(h) @ self.oh_basis
            coef_hat[:, self.oh_idx] = coef_hat[:, self.oh_idx] + oh_resid

        if self.has_continuum_head:
            ctx_block = cond[:, -2 * self.n_ctx :]
            continuum_delta = self.ctx_continuum_backbone(ctx_block) @ self.continuum_basis
            coef_hat[:, self.continuum_idx] = coef_hat[:, self.continuum_idx] + continuum_delta

        return coef_hat


def _kl_gaussian(mu_q, logvar_q, mu_p, logvar_p):
    var_q = torch.exp(logvar_q)
    var_p = torch.exp(logvar_p)
    kl = 0.5 * (
        logvar_p - logvar_q + (var_q + (mu_q - mu_p) ** 2) / torch.clamp(var_p, min=1e-8) - 1.0
    )
    return torch.sum(kl, dim=-1)


def _extract_component_indices_from_names(coef_names, prefixes):
    pfx = tuple(str(p).lower() for p in prefixes)
    out = []
    for i, name in enumerate(coef_names):
        lname = str(name).lower()
        if lname.startswith(pfx):
            out.append(i)
    return np.asarray(out, dtype=np.int64)


def _select_continuum_indices(coef_names):
    """Return coefficient indices for continuum/diffuse families only."""
    keep = []
    for i, name in enumerate(coef_names):
        lname = str(name).lower()
        is_cont_or_diffuse = any(k in lname for k in ("moon_bs", "diffuse", "continuum", "ho2", "feo", "o2"))
        is_excluded = any(k in lname for k in ("oh", "atom"))
        if is_cont_or_diffuse and not is_excluded:
            keep.append(i)
    return np.asarray(keep, dtype=np.int64)


def _smoothness_penalty(x, idx, order=2):
    if idx is None or len(idx) < 3:
        return torch.tensor(0.0, device=x.device)
    sel = x[:, idx]
    if order == 2 and sel.shape[1] >= 3:
        d2 = sel[:, 2:] - 2.0 * sel[:, 1:-1] + sel[:, :-2]
        return torch.mean(torch.abs(d2))
    if sel.shape[1] >= 2:
        d1 = sel[:, 1:] - sel[:, :-1]
        return torch.mean(torch.abs(d1))
    return torch.tensor(0.0, device=x.device)


def _cvae_loss_stable_triplet(
    coef_hat,
    coef_true,
    mu_q,
    logvar_q,
    mu_p,
    logvar_p,
    beta,
    latent_l2_weight=2e-4,
    coef_hat_ctx=None,
    context_pred_weight=0.5,
    continuum_idx=None,
    continuum_pred_weight=2.0,
    spline_idx=None,
    smoothness_weight=2e-4,
    oh_idx=None,
    oh_smoothness_weight=0.0,
    oh_tail_weight=0.0,
    oh_tail_gamma=2.0,
    oh_tail_ref=None,
):
    recon = F.smooth_l1_loss(coef_hat, coef_true)
    kl = _kl_gaussian(mu_q, logvar_q, mu_p, logvar_p).mean()
    latent_l2 = torch.mean(mu_q ** 2)

    spline_smooth = _smoothness_penalty(coef_hat, spline_idx, order=2)
    oh_smooth = _smoothness_penalty(coef_hat, oh_idx, order=1)

    oh_tail_penalty = torch.tensor(0.0, device=coef_hat.device)
    if (
        oh_tail_weight > 0.0
        and oh_idx is not None
        and len(oh_idx) > 0
        and oh_tail_ref is not None
    ):
        oh_pred = coef_hat[:, oh_idx]
        oh_true = coef_true[:, oh_idx]
        oh_ref = oh_tail_ref.to(coef_true.device).view(1, -1)
        oh_excess = torch.relu(oh_true - oh_ref)
        oh_weights = 1.0 + float(oh_tail_gamma) * oh_excess
        oh_tail_penalty = (F.smooth_l1_loss(oh_pred, oh_true, reduction="none") * oh_weights).mean()

    ctx_pred = torch.tensor(0.0, device=coef_hat.device)
    cont_pred = torch.tensor(0.0, device=coef_hat.device)
    if coef_hat_ctx is not None:
        ctx_pred = F.smooth_l1_loss(coef_hat_ctx, coef_true)
        if continuum_idx is not None and len(continuum_idx) > 0:
            cont_pred = F.smooth_l1_loss(coef_hat_ctx[:, continuum_idx], coef_true[:, continuum_idx])

    loss = (
        recon
        + float(beta) * kl
        + float(latent_l2_weight) * latent_l2
        + float(smoothness_weight) * spline_smooth
        + float(oh_smoothness_weight) * oh_smooth
        + float(oh_tail_weight) * oh_tail_penalty
        + float(context_pred_weight) * ctx_pred
        + float(continuum_pred_weight) * cont_pred
    )
    return (
        loss,
        recon.detach(),
        kl.detach(),
        spline_smooth.detach(),
        oh_smooth.detach(),
        ctx_pred.detach(),
        cont_pred.detach(),
    )


def train_coeff_prediction_cvae(
    filtered,
    n_epochs=200,
    batch_size=256,
    lr=1e-3,
    z_dim=8,
    beta_max=0.3,
    beta_warmup_epochs=40,
    latent_l2_weight=2e-4,
    context_pred_weight=0.5,
    continuum_pred_weight=2.0,
    spline_rank=4,
    smoothness_weight=2e-4,
    spline_prefixes=("moon_bs",),
    oh_rank=8,
    oh_smoothness_weight=0.0,
    oh_prefixes=("oh", "oh_"),
    oh_tail_weight=0.20,
    oh_tail_percentile=95.0,
    oh_tail_gamma=2.0,
    continuum_rank=8,
    grad_clip=1.0,
    seed=42,
):
    _set_reproducibility(seed)

    coef_near = np.asarray(filtered["coef_near"], dtype=np.float32)
    coef_far = np.asarray(filtered["coef_far"], dtype=np.float32)
    coef_sci = np.asarray(filtered["coef_sci"], dtype=np.float32)
    ctx_near = np.asarray(filtered["ctx_near"], dtype=np.float32)
    ctx_far = np.asarray(filtered["ctx_far"], dtype=np.float32)
    ctx_sci = np.asarray(filtered["ctx_sci"], dtype=np.float32)

    n = coef_near.shape[0]
    train_idx, val_idx, test_idx = split_indices(n, seed=seed)

    near_model = _coef_to_model_space(coef_near)
    far_model = _coef_to_model_space(coef_far)
    sci_model = _coef_to_model_space(coef_sci)

    coef_scaler = RobustScaler().fit(np.vstack([near_model[train_idx], far_model[train_idx], sci_model[train_idx]]))
    ctx_scaler = RobustScaler().fit(np.vstack([ctx_near[train_idx], ctx_far[train_idx], ctx_sci[train_idx]]))

    near_n = np.clip(coef_scaler.transform(near_model), -25.0, 25.0)
    far_n = np.clip(coef_scaler.transform(far_model), -25.0, 25.0)
    sci_n = np.clip(coef_scaler.transform(sci_model), -25.0, 25.0)

    near_ctx_n = np.clip(ctx_scaler.transform(ctx_near), -25.0, 25.0)
    far_ctx_n = np.clip(ctx_scaler.transform(ctx_far), -25.0, 25.0)
    sci_ctx_n = np.clip(ctx_scaler.transform(ctx_sci), -25.0, 25.0)
    delta_ctx_n = sci_ctx_n - 0.5 * (near_ctx_n + far_ctx_n)

    tr_loader = _make_loader(
        near_n[train_idx], far_n[train_idx], delta_ctx_n[train_idx], sci_ctx_n[train_idx], sci_n[train_idx],
        batch_size=batch_size,
        shuffle=True,
    )
    va_loader = _make_loader(
        near_n[val_idx], far_n[val_idx], delta_ctx_n[val_idx], sci_ctx_n[val_idx], sci_n[val_idx],
        batch_size=batch_size,
        shuffle=False,
    )

    if torch.cuda.is_available():
        device = "cuda"
    elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available() and batch_size > 256:
        device = "mps"
    else:
        device = "cpu"

    coef_group_ids_np, coef_group_names = _build_physics_group_ids(filtered["coef_names"])
    coef_group_ids_t = torch.from_numpy(coef_group_ids_np)

    continuum_idx_np = _select_continuum_indices(filtered["coef_names"])
    spline_idx_np = _extract_component_indices_from_names(filtered["coef_names"], spline_prefixes)
    oh_idx_np = _extract_component_indices_from_names(filtered["coef_names"], oh_prefixes)

    print(f"Spline head targets: n={spline_idx_np.size}, rank={int(spline_rank)}")
    print(f"OH head targets: n={oh_idx_np.size}, rank={int(oh_rank)}")
    print(f"Continuum context targets: n={continuum_idx_np.size}, rank={int(continuum_rank)}")

    oh_tail_ref_t = None
    if oh_idx_np.size > 0 and oh_tail_weight > 0.0:
        oh_ref = np.percentile(sci_n[train_idx][:, oh_idx_np], float(oh_tail_percentile), axis=0).astype(np.float32)
        oh_tail_ref_t = torch.from_numpy(oh_ref)
        print(
            f"OH-tail robust loss active: weight={oh_tail_weight:.3f}, "
            f"percentile={oh_tail_percentile:.1f}, gamma={oh_tail_gamma:.2f}, n={oh_idx_np.size}"
        )

    model = ConditionalCoeffVAE(
        n_coef=near_n.shape[1],
        n_ctx=sci_ctx_n.shape[1],
        z_dim=int(z_dim),
        hidden=256,
        coef_group_ids=coef_group_ids_t,
        n_groups=len(coef_group_names),
        group_emb_dim=8,
        spline_indices=spline_idx_np,
        spline_rank=int(spline_rank),
        oh_indices=oh_idx_np,
        oh_rank=int(oh_rank),
        continuum_indices=continuum_idx_np,
        continuum_rank=int(continuum_rank),
    ).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)

    best = {"val_loss": np.inf, "state": None, "epoch": -1}
    history = []

    continuum_idx_t = torch.from_numpy(continuum_idx_np).to(device) if continuum_idx_np.size > 0 else None
    spline_idx_t = torch.from_numpy(spline_idx_np).to(device) if spline_idx_np.size > 0 else None
    oh_idx_t = torch.from_numpy(oh_idx_np).to(device) if oh_idx_np.size > 0 else None

    for epoch in range(1, n_epochs + 1):
        beta = float(beta_max) * min(1.0, epoch / max(int(beta_warmup_epochs), 1))

        model.train()
        tr_loss = 0.0
        tr_recon = 0.0
        tr_kl = 0.0
        tr_spline = 0.0
        tr_oh = 0.0
        tr_ctx = 0.0
        tr_cont = 0.0
        tr_batches = 0
        for near_b, far_b, dctx_b, scictx_b, target_b in tr_loader:
            near_b = near_b.to(device)
            far_b = far_b.to(device)
            dctx_b = dctx_b.to(device)
            scictx_b = scictx_b.to(device)
            target_b = target_b.to(device)

            mu_q, logvar_q, cond_b = model.encode(near_b, far_b, dctx_b, scictx_b, target_b)
            mu_p, logvar_p = model.prior(cond_b)
            eps = torch.randn_like(mu_q)
            z = mu_q + torch.exp(0.5 * logvar_q) * eps
            pred_b = model.decode(cond_b, z)
            coef_hat_ctx = model.decode(cond_b, mu_p)

            loss, recon, kl, spline_smooth, oh_smooth, ctx_pred, cont_pred = _cvae_loss_stable_triplet(
                pred_b,
                target_b,
                mu_q,
                logvar_q,
                mu_p,
                logvar_p,
                beta=beta,
                latent_l2_weight=latent_l2_weight,
                coef_hat_ctx=coef_hat_ctx,
                context_pred_weight=context_pred_weight,
                continuum_idx=continuum_idx_t,
                continuum_pred_weight=continuum_pred_weight,
                spline_idx=spline_idx_t,
                smoothness_weight=smoothness_weight,
                oh_idx=oh_idx_t,
                oh_smoothness_weight=oh_smoothness_weight,
                oh_tail_weight=oh_tail_weight,
                oh_tail_gamma=oh_tail_gamma,
                oh_tail_ref=oh_tail_ref_t,
            )

            if not torch.isfinite(loss):
                continue

            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), float(grad_clip))
            opt.step()

            tr_loss += float(loss.item())
            tr_recon += float(recon.item())
            tr_kl += float(kl.item())
            tr_spline += float(spline_smooth.item())
            tr_oh += float(oh_smooth.item())
            tr_ctx += float(ctx_pred.item())
            tr_cont += float(cont_pred.item())
            tr_batches += 1

        model.eval()
        va_loss = 0.0
        va_recon = 0.0
        va_kl = 0.0
        va_spline = 0.0
        va_oh = 0.0
        va_ctx = 0.0
        va_cont = 0.0
        va_batches = 0
        with torch.no_grad():
            for near_b, far_b, dctx_b, scictx_b, target_b in va_loader:
                near_b = near_b.to(device)
                far_b = far_b.to(device)
                dctx_b = dctx_b.to(device)
                scictx_b = scictx_b.to(device)
                target_b = target_b.to(device)

                mu_q, logvar_q, cond_b = model.encode(near_b, far_b, dctx_b, scictx_b, target_b)
                mu_p, logvar_p = model.prior(cond_b)
                pred_b = model.decode(cond_b, mu_q)
                coef_hat_ctx = model.decode(cond_b, mu_p)

                loss, recon, kl, spline_smooth, oh_smooth, ctx_pred, cont_pred = _cvae_loss_stable_triplet(
                    pred_b,
                    target_b,
                    mu_q,
                    logvar_q,
                    mu_p,
                    logvar_p,
                    beta=beta,
                    latent_l2_weight=latent_l2_weight,
                    coef_hat_ctx=coef_hat_ctx,
                    context_pred_weight=context_pred_weight,
                    continuum_idx=continuum_idx_t,
                    continuum_pred_weight=continuum_pred_weight,
                    spline_idx=spline_idx_t,
                    smoothness_weight=smoothness_weight,
                    oh_idx=oh_idx_t,
                    oh_smoothness_weight=oh_smoothness_weight,
                    oh_tail_weight=oh_tail_weight,
                    oh_tail_gamma=oh_tail_gamma,
                    oh_tail_ref=oh_tail_ref_t,
                )

                if not torch.isfinite(loss):
                    continue

                va_loss += float(loss.item())
                va_recon += float(recon.item())
                va_kl += float(kl.item())
                va_spline += float(spline_smooth.item())
                va_oh += float(oh_smooth.item())
                va_ctx += float(ctx_pred.item())
                va_cont += float(cont_pred.item())
                va_batches += 1

        tr_loss_m = tr_loss / max(tr_batches, 1)
        tr_recon_m = tr_recon / max(tr_batches, 1)
        tr_kl_m = tr_kl / max(tr_batches, 1)
        tr_spline_m = tr_spline / max(tr_batches, 1)
        tr_oh_m = tr_oh / max(tr_batches, 1)
        tr_ctx_m = tr_ctx / max(tr_batches, 1)
        tr_cont_m = tr_cont / max(tr_batches, 1)

        va_loss_m = va_loss / max(va_batches, 1)
        va_recon_m = va_recon / max(va_batches, 1)
        va_kl_m = va_kl / max(va_batches, 1)
        va_spline_m = va_spline / max(va_batches, 1)
        va_oh_m = va_oh / max(va_batches, 1)
        va_ctx_m = va_ctx / max(va_batches, 1)
        va_cont_m = va_cont / max(va_batches, 1)

        history.append(
            {
                "epoch": epoch,
                "beta": beta,
                "train_loss": tr_loss_m,
                "train_recon": tr_recon_m,
                "train_kl": tr_kl_m,
                "train_spline_smooth": tr_spline_m,
                "train_oh_smooth": tr_oh_m,
                "train_ctx_pred": tr_ctx_m,
                "train_cont_pred": tr_cont_m,
                "val_loss": va_loss_m,
                "val_recon": va_recon_m,
                "val_kl": va_kl_m,
                "val_spline_smooth": va_spline_m,
                "val_oh_smooth": va_oh_m,
                "val_ctx_pred": va_ctx_m,
                "val_cont_pred": va_cont_m,
            }
        )

        if va_loss_m < best["val_loss"]:
            best["val_loss"] = va_loss_m
            best["state"] = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best["epoch"] = epoch

        if epoch == 1 or epoch % 10 == 0:
            print(
                f"[cvae] epoch={epoch:03d} beta={beta:.4f} "
                f"train={tr_loss_m:.5f} val={va_loss_m:.5f} "
                f"recon={va_recon_m:.5f} kl={va_kl_m:.5f} "
                f"spline={va_spline_m:.5f} oh={va_oh_m:.5f} "
                f"ctx={va_ctx_m:.5f} cont={va_cont_m:.5f}"
            )

    if best["state"] is not None:
        model.load_state_dict(best["state"])

    return {
        "model": model,
        "device": device,
        "coef_scaler": coef_scaler,
        "ctx_scaler": ctx_scaler,
        "history": history,
        "best_epoch": best["epoch"],
        "best_val_loss": best["val_loss"],
        "train_idx": train_idx,
        "val_idx": val_idx,
        "test_idx": test_idx,
        "coef_names": list(filtered["coef_names"]),
        "ctx_names": list(filtered["ctx_names"]),
        "coef_group_names": coef_group_names,
        "continuum_idx": continuum_idx_np,
        "continuum_rank": int(continuum_rank),
        "spline_idx": spline_idx_np,
        "spline_rank": int(spline_rank),
        "oh_idx": oh_idx_np,
        "oh_rank": int(oh_rank),
        "z_dim": int(z_dim),
        "beta_max": float(beta_max),
    }


def predict_sci_coefficients_cvae(
    cvae_artifacts,
    coef_near_phys,
    coef_far_phys,
    ctx_near_phys,
    ctx_far_phys,
    ctx_sci_phys,
    deterministic=True,
    n_samples=16,
    seed=42,
    return_samples=False,
):
    model = cvae_artifacts["model"]
    device = cvae_artifacts["device"]
    coef_scaler = cvae_artifacts["coef_scaler"]
    ctx_scaler = cvae_artifacts["ctx_scaler"]

    coef_near_phys = np.asarray(coef_near_phys, dtype=np.float32)
    coef_far_phys = np.asarray(coef_far_phys, dtype=np.float32)
    ctx_near_phys = np.asarray(ctx_near_phys, dtype=np.float32)
    ctx_far_phys = np.asarray(ctx_far_phys, dtype=np.float32)
    ctx_sci_phys = np.asarray(ctx_sci_phys, dtype=np.float32)

    if coef_near_phys.ndim == 1:
        coef_near_phys = coef_near_phys[None, :]
    if coef_far_phys.ndim == 1:
        coef_far_phys = coef_far_phys[None, :]
    if ctx_near_phys.ndim == 1:
        ctx_near_phys = ctx_near_phys[None, :]
    if ctx_far_phys.ndim == 1:
        ctx_far_phys = ctx_far_phys[None, :]
    if ctx_sci_phys.ndim == 1:
        ctx_sci_phys = ctx_sci_phys[None, :]

    near_n = np.clip(coef_scaler.transform(_coef_to_model_space(coef_near_phys)), -25.0, 25.0).astype(np.float32)
    far_n = np.clip(coef_scaler.transform(_coef_to_model_space(coef_far_phys)), -25.0, 25.0).astype(np.float32)

    near_ctx_n = np.clip(ctx_scaler.transform(ctx_near_phys), -25.0, 25.0).astype(np.float32)
    far_ctx_n = np.clip(ctx_scaler.transform(ctx_far_phys), -25.0, 25.0).astype(np.float32)
    sci_ctx_n = np.clip(ctx_scaler.transform(ctx_sci_phys), -25.0, 25.0).astype(np.float32)
    delta_ctx_n = sci_ctx_n - 0.5 * (near_ctx_n + far_ctx_n)

    with torch.no_grad():
        near_t = torch.from_numpy(near_n).to(device)
        far_t = torch.from_numpy(far_n).to(device)
        dctx_t = torch.from_numpy(delta_ctx_n).to(device)
        scictx_t = torch.from_numpy(sci_ctx_n).to(device)

        cond_t = model._build_cond(near_t, far_t, dctx_t, scictx_t)
        mu_p, logvar_p = model.prior(cond_t)

        if deterministic:
            pred_n_t = model.decode(cond_t, mu_p)
            pred_n = pred_n_t.cpu().numpy()
            pred_model = coef_scaler.inverse_transform(pred_n)
            pred_phys = _coef_from_model_space(pred_model)
            return pred_phys

        gen = torch.Generator(device=device)
        gen.manual_seed(int(seed))
        samples_n = []
        std_p = torch.exp(0.5 * logvar_p)
        for _ in range(int(n_samples)):
            eps = torch.randn(mu_p.shape, generator=gen, device=device)
            z = mu_p + std_p * eps
            pred_s = model.decode(cond_t, z)
            samples_n.append(pred_s.cpu().numpy())

    samples_n = np.asarray(samples_n, dtype=np.float32)
    samples_model = coef_scaler.inverse_transform(samples_n.reshape(-1, samples_n.shape[-1])).reshape(samples_n.shape)
    samples_phys = _coef_from_model_space(samples_model)
    pred_phys_mean = np.mean(samples_phys, axis=0)

    if return_samples:
        return pred_phys_mean, samples_phys
    return pred_phys_mean

In [ ]:
# Train cVAE and compare against the current direct regressor
required = [
    "filtered_triplet",
    "train_coeff_prediction_cvae",
    "predict_sci_coefficients_cvae",
    "predict_sci_coefficients_geo_weighted",
    "coeff_model_artifacts",
]
missing = [k for k in required if k not in globals()]
if missing:
    raise RuntimeError("Run prerequisite cells first. Missing: " + ", ".join(missing))

# ND-inspired stable cVAE tuning for this triplet problem.
cvae_artifacts = train_coeff_prediction_cvae(
    filtered_triplet,
    n_epochs=200,
    batch_size=256,
    lr=1e-3,
    z_dim=8,
    beta_max=0.3,
    beta_warmup_epochs=80,
    latent_l2_weight=2e-4,
    context_pred_weight=0.5,
    continuum_pred_weight=2.0,
    spline_rank=4,
    smoothness_weight=2e-4,
    spline_prefixes=("moon_bs",),
    oh_rank=8,
    oh_smoothness_weight=0.0,
    oh_prefixes=("oh", "oh_"),
    oh_tail_weight=0.20,
    oh_tail_percentile=95.0,
    oh_tail_gamma=2.0,
    continuum_rank=8,
    grad_clip=1.0,
    seed=42,
)
print(
    f"Best cVAE epoch: {cvae_artifacts['best_epoch']} "
    f"| best val score: {cvae_artifacts['best_val_loss']:.6f}"
)

test_idx_cvae = np.asarray(cvae_artifacts["test_idx"], dtype=int)
coef_true_cvae = np.asarray(filtered_triplet["coef_sci"][test_idx_cvae], dtype=np.float32)

coef_pred_direct = predict_sci_coefficients_geo_weighted(
    coeff_model_artifacts,
    coef_near_phys=filtered_triplet["coef_near"][test_idx_cvae],
    coef_far_phys=filtered_triplet["coef_far"][test_idx_cvae],
    ctx_near_phys=filtered_triplet["ctx_near"][test_idx_cvae],
    ctx_far_phys=filtered_triplet["ctx_far"][test_idx_cvae],
    ctx_sci_phys=filtered_triplet["ctx_sci"][test_idx_cvae],
)

coef_pred_cvae_det = predict_sci_coefficients_cvae(
    cvae_artifacts,
    coef_near_phys=filtered_triplet["coef_near"][test_idx_cvae],
    coef_far_phys=filtered_triplet["coef_far"][test_idx_cvae],
    ctx_near_phys=filtered_triplet["ctx_near"][test_idx_cvae],
    ctx_far_phys=filtered_triplet["ctx_far"][test_idx_cvae],
    ctx_sci_phys=filtered_triplet["ctx_sci"][test_idx_cvae],
    deterministic=True,
)

coef_pred_cvae_mc, coef_pred_cvae_samples = predict_sci_coefficients_cvae(
    cvae_artifacts,
    coef_near_phys=filtered_triplet["coef_near"][test_idx_cvae],
    coef_far_phys=filtered_triplet["coef_far"][test_idx_cvae],
    ctx_near_phys=filtered_triplet["ctx_near"][test_idx_cvae],
    ctx_far_phys=filtered_triplet["ctx_far"][test_idx_cvae],
    ctx_sci_phys=filtered_triplet["ctx_sci"][test_idx_cvae],
    deterministic=False,
    n_samples=16,
    seed=42,
    return_samples=True,
)


def _metric_row(y_true, y_pred, model_name):
    rmse = np.sqrt(np.mean((y_pred - y_true) ** 2, axis=0))
    mae = np.mean(np.abs(y_pred - y_true), axis=0)
    corr = []
    for j in range(y_true.shape[1]):
        x = y_true[:, j]
        y = y_pred[:, j]
        if np.std(x) < 1e-12 or np.std(y) < 1e-12:
            corr.append(np.nan)
        else:
            corr.append(float(np.corrcoef(x, y)[0, 1]))
    corr = np.asarray(corr)
    return {
        "model": model_name,
        "mean_rmse": float(np.nanmean(rmse)),
        "median_rmse": float(np.nanmedian(rmse)),
        "mean_mae": float(np.nanmean(mae)),
        "mean_corr": float(np.nanmean(corr)),
        "median_corr": float(np.nanmedian(corr)),
    }


rows_cmp = [
    _metric_row(coef_true_cvae, coef_pred_direct, "direct_mlp"),
    _metric_row(coef_true_cvae, coef_pred_cvae_det, "cvae_det_prior_mean"),
    _metric_row(coef_true_cvae, coef_pred_cvae_mc, "cvae_mc_mean_16"),
]
cmp_df = pd.DataFrame(rows_cmp)

print("\nCoefficient prediction comparison on shared test split:")
print(cmp_df.to_string(index=False, float_format=lambda v: f"{v:.6g}"))

cmp_plot_df = cmp_df.melt(
    id_vars=["model"],
    value_vars=["mean_rmse", "median_rmse", "mean_mae", "mean_corr", "median_corr"],
    var_name="metric",
    value_name="value",
)
fig_cmp = px.bar(
    cmp_plot_df,
    x="metric",
    y="value",
    color="model",
    barmode="group",
    title="Direct model vs cVAE comparison",
)
fig_cmp.update_layout(template="plotly_white", height=450)
fig_cmp.show()

[ERROR]: Traceback (most recent call last):
  File "/opt/miniconda/envs/lvmdrp/lib/python3.12/site-packages/IPython/core/interactiveshell.py", line 3577, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/var/folders/31/fxk1ql6s5bx7q3kh6kwpf8v8c5vp86/T/ipykernel_32922/4178733419.py", line 14, in <module>
    cvae_artifacts = train_coeff_prediction_cvae(
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: train_coeff_prediction_cvae() got an unexpected keyword argument 'kl_warmup_epochs'



## Coefficient Model Diagnostics

These diagnostics summarize grouped coefficient behavior versus context for near/far/true-science/predicted-science coefficient summaries.

In [25]:
# Relationship plots: grouped coefficient-vs-context structure
import pandas as pd
import plotly.express as px

required = ["coeff_model_artifacts", "filtered_triplet", "predict_sci_coefficients_geo_weighted"]
missing = [k for k in required if k not in globals()]
if missing:
    raise RuntimeError("Run the coefficient training cell first. Missing: " + ", ".join(missing))

coef_near_all = np.asarray(filtered_triplet["coef_near"], dtype=np.float32)
coef_far_all = np.asarray(filtered_triplet["coef_far"], dtype=np.float32)
coef_sci_all = np.asarray(filtered_triplet["coef_sci"], dtype=np.float32)
ctx_near_all = np.asarray(filtered_triplet["ctx_near"], dtype=np.float32)
ctx_far_all = np.asarray(filtered_triplet["ctx_far"], dtype=np.float32)
ctx_sci_all = np.asarray(filtered_triplet["ctx_sci"], dtype=np.float32)
coef_names_all = [str(n) for n in filtered_triplet["coef_names"]]
ctx_names_all = [str(n) for n in filtered_triplet["ctx_names"]]

coef_pred_all = predict_sci_coefficients_geo_weighted(
    coeff_model_artifacts,
    coef_near_phys=coef_near_all,
    coef_far_phys=coef_far_all,
    ctx_near_phys=ctx_near_all,
    ctx_far_phys=ctx_far_all,
    ctx_sci_phys=ctx_sci_all,
)

coef_name_l = [n.lower() for n in coef_names_all]

def _group_idx(names_l):
    groups = {
        "diffuse_continuum_median": [
            i for i, n in enumerate(names_l)
            if (n.startswith("ho2") or n.startswith("feo") or n.startswith("o2") or "diffuse" in n or "continuum" in n)
        ],
        "oh_median": [i for i, n in enumerate(names_l) if n.startswith("oh")],
        "atomic_median": [i for i, n in enumerate(names_l) if n.startswith("atom")],
    }
    return {k: np.asarray(v, dtype=int) for k, v in groups.items() if len(v) > 0}

group_idx = _group_idx(coef_name_l)
if len(group_idx) == 0:
    raise RuntimeError("No coefficient groups found for relationship diagnostics")

rows_rel = []
max_points = 6000
n_rows = coef_sci_all.shape[0]
if n_rows > max_points:
    rng = np.random.default_rng(42)
    use = np.sort(rng.choice(n_rows, size=max_points, replace=False))
else:
    use = np.arange(n_rows)

for gname, idx in group_idx.items():
    near_g = np.nanmedian(coef_near_all[use][:, idx], axis=1)
    far_g = np.nanmedian(coef_far_all[use][:, idx], axis=1)
    sci_true_g = np.nanmedian(coef_sci_all[use][:, idx], axis=1)
    sci_pred_g = np.nanmedian(coef_pred_all[use][:, idx], axis=1)

    for j, cname in enumerate(ctx_names_all):
        x = ctx_sci_all[use, j]
        rows_rel.append(pd.DataFrame({"group": gname, "context_param": cname, "context_value": x, "coef_value": near_g, "series": "near"}))
        rows_rel.append(pd.DataFrame({"group": gname, "context_param": cname, "context_value": x, "coef_value": far_g, "series": "far"}))
        rows_rel.append(pd.DataFrame({"group": gname, "context_param": cname, "context_value": x, "coef_value": sci_true_g, "series": "sci_true"}))
        rows_rel.append(pd.DataFrame({"group": gname, "context_param": cname, "context_value": x, "coef_value": sci_pred_g, "series": "sci_pred"}))

rel_df = pd.concat(rows_rel, ignore_index=True)

fig_rel = px.scatter(
    rel_df,
    x="context_value",
    y="coef_value",
    color="series",
    facet_col="context_param",
    facet_row="group",
    opacity=0.20,
    render_mode="webgl",
    title="Grouped coefficient relationships vs context (near/far/true/pred)",
    color_discrete_map={
        "near": "#1f77b4",
        "far": "#9467bd",
        "sci_true": "#2ca02c",
        "sci_pred": "#d62728",
    },
)
fig_rel.for_each_annotation(
    lambda a: a.update(text=a.text.replace("group=", "").replace("context_param=", ""))
)
fig_rel.update_xaxes(matches=None)
fig_rel.update_yaxes(matches=None)
fig_rel.update_layout(template="plotly_white", height=max(750, 210 * len(group_idx)))
fig_rel.show()

In [26]:
# cVAE latent-space diagnostics: latent dimensions vs context and selected coefficients
import pandas as pd
import plotly.express as px

required = ["cvae_artifacts", "filtered_triplet"]
missing = [k for k in required if k not in globals()]
if missing:
    raise RuntimeError("Run cVAE training cell first. Missing: " + ", ".join(missing))

model = cvae_artifacts["model"]
device = cvae_artifacts["device"]
coef_scaler = cvae_artifacts["coef_scaler"]
ctx_scaler = cvae_artifacts["ctx_scaler"]

test_idx = np.asarray(cvae_artifacts["test_idx"], dtype=int)

coef_near = np.asarray(filtered_triplet["coef_near"][test_idx], dtype=np.float32)
coef_far = np.asarray(filtered_triplet["coef_far"][test_idx], dtype=np.float32)
coef_sci = np.asarray(filtered_triplet["coef_sci"][test_idx], dtype=np.float32)
ctx_near = np.asarray(filtered_triplet["ctx_near"][test_idx], dtype=np.float32)
ctx_far = np.asarray(filtered_triplet["ctx_far"][test_idx], dtype=np.float32)
ctx_sci = np.asarray(filtered_triplet["ctx_sci"][test_idx], dtype=np.float32)

coef_names = [str(n) for n in filtered_triplet["coef_names"]]
ctx_names = [str(n) for n in filtered_triplet["ctx_names"]]

near_n = np.clip(coef_scaler.transform(_coef_to_model_space(coef_near)), -25.0, 25.0).astype(np.float32)
far_n = np.clip(coef_scaler.transform(_coef_to_model_space(coef_far)), -25.0, 25.0).astype(np.float32)
near_ctx_n = np.clip(ctx_scaler.transform(ctx_near), -25.0, 25.0).astype(np.float32)
far_ctx_n = np.clip(ctx_scaler.transform(ctx_far), -25.0, 25.0).astype(np.float32)
sci_ctx_n = np.clip(ctx_scaler.transform(ctx_sci), -25.0, 25.0).astype(np.float32)
delta_ctx_n = sci_ctx_n - 0.5 * (near_ctx_n + far_ctx_n)

with torch.no_grad():
    near_t = torch.from_numpy(near_n).to(device)
    far_t = torch.from_numpy(far_n).to(device)
    dctx_t = torch.from_numpy(delta_ctx_n).to(device)
    scictx_t = torch.from_numpy(sci_ctx_n).to(device)

    cond_t = model._build_cond(near_t, far_t, dctx_t, scictx_t)
    mu_t, _ = model.prior(cond_t)
    coef_hat_n = model.decode(cond_t, mu_t).cpu().numpy()

coef_hat_model = coef_scaler.inverse_transform(coef_hat_n)
coef_hat_phys = _coef_from_model_space(coef_hat_model)


def _robust_bounds(arr, q_low=1.0, q_high=99.0, pad_frac=0.05):
    arr = np.asarray(arr, dtype=np.float64)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return (-1.0, 1.0)

    lo, hi = np.nanpercentile(arr, [q_low, q_high])
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        lo = np.nanmin(arr)
        hi = np.nanmax(arr)

    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        c = float(arr[0])
        lo, hi = c - 0.5, c + 0.5

    span = max(hi - lo, 1e-6)
    pad = pad_frac * span
    return lo - pad, hi + pad


def _clip_to_bounds(arr, bounds):
    return np.clip(np.asarray(arr, dtype=np.float64), bounds[0], bounds[1])


mu_np = mu_t.cpu().numpy()

# Pick latent dims with highest variance for readability.
latent_var = np.nanvar(mu_np, axis=0)
n_latent_plot = int(min(6, mu_np.shape[1]))
latent_sel = np.argsort(latent_var)[-n_latent_plot:][::-1]

# Fixed coefficient subset: diffuse/continuum family only, excluding moon_bs, OH and atom.
coef_name_l = [n.lower() for n in coef_names]
coef_sel_list = []
for j, n in enumerate(coef_name_l):
    is_diffuse_continuum = (
        ("continuum" in n)
        or ("diffuse" in n)
        or n.startswith("feo")
        or n.startswith("ho2")
        or n.startswith("o2")
    )
    is_excluded = n.startswith("moon_bs") or n.startswith("oh") or n.startswith("atom")
    if is_diffuse_continuum and not is_excluded:
        coef_sel_list.append(j)

if len(coef_sel_list) == 0:
    raise RuntimeError("No diffuse/continuum coefficients found (with moon_bs/atom/OH excluded).")

n_coef_plot = int(min(4, len(coef_sel_list)))
coef_sel = np.asarray(coef_sel_list[:n_coef_plot], dtype=int)

n_ctx_plot = int(min(4, len(ctx_names)))
ctx_sel = np.arange(n_ctx_plot, dtype=int)

max_points = 4000
n_rows = mu_np.shape[0]
if n_rows > max_points:
    rng = np.random.default_rng(42)
    use = np.sort(rng.choice(n_rows, size=max_points, replace=False))
else:
    use = np.arange(n_rows)

rows = []
for ld in latent_sel:
    z = mu_np[use, ld]
    z_bounds = _robust_bounds(z)

    for cj in ctx_sel:
        x = ctx_sci[use, cj]
        x_bounds = _robust_bounds(x)
        rows.append(pd.DataFrame({
            "latent_dim": f"z{ld}",
            "variable": ctx_names[cj],
            "var_type": "context",
            "x_value": _clip_to_bounds(x, x_bounds),
            "z_value": _clip_to_bounds(z, z_bounds),
        }))

    for kj in coef_sel:
        x_true = coef_sci[use, kj]
        x_pred = coef_hat_phys[use, kj]
        x_bounds = _robust_bounds(np.concatenate([x_true, x_pred]))

        rows.append(pd.DataFrame({
            "latent_dim": f"z{ld}",
            "variable": coef_names[kj],
            "var_type": "coef_true",
            "x_value": _clip_to_bounds(x_true, x_bounds),
            "z_value": _clip_to_bounds(z, z_bounds),
        }))
        rows.append(pd.DataFrame({
            "latent_dim": f"z{ld}",
            "variable": coef_names[kj],
            "var_type": "coef_pred",
            "x_value": _clip_to_bounds(x_pred, x_bounds),
            "z_value": _clip_to_bounds(z, z_bounds),
        }))

latent_df = pd.concat(rows, ignore_index=True)

fig_latent = px.scatter(
    latent_df,
    x="x_value",
    y="z_value",
    color="var_type",
    facet_row="latent_dim",
    facet_col="variable",
    opacity=0.24,
    render_mode="webgl",
    title="cVAE latent-space grid: latent dims vs context and selected coefficients",
    color_discrete_map={
        "context": "#1f77b4",
        "coef_true": "#2ca02c",
        "coef_pred": "#d62728",
    },
)
fig_latent.for_each_annotation(
    lambda a: a.update(text=a.text.replace("latent_dim=", "").replace("variable=", ""))
)
fig_latent.update_xaxes(matches=None)
fig_latent.update_yaxes(matches=None)
fig_latent.update_layout(template="plotly_white", height=max(820, 230 * n_latent_plot))
fig_latent.show()

## Full-Spectrum Test From Predicted Coefficients

> Primary workflow: predict SCI decomposition coefficients, then reconstruct SCI spectrum with the decomposition forward model.

Data usage policy:
- `*_meta_coef.fits` products are used only for **training** the coefficient model.
- The four `*_every10*.fits` files are used only for **testing** and are never seen during training.

This cell reconstructs only the requested row from the every10 set:
1. Predict science coefficients for that row.
2. Reconstruct the science spectrum using the row wavelength grid and LSF.
3. Plot reconstructed vs observed spectra and relative residuals.


In [27]:
# Full-spectrum reconstruction test for a single requested row using cVAE coefficients
import plotly.graph_objects as go

EVERY10_INPUT = "lvmsframe_median_stack_1.2.1_every10.fits"
EVERY10_NEAR = "lvmsframe_median_stack_1.2.1_every10_decomp_sky1.fits"
EVERY10_FAR = "lvmsframe_median_stack_1.2.1_every10_decomp_sky2.fits"
EVERY10_SCI = "lvmsframe_median_stack_1.2.1_every10_decomp_sci.fits"

# Set the row to reconstruct and inspect.
REQUESTED_ROW = 1000

# cVAE inference mode for reconstruction.
CVAE_DETERMINISTIC = True
CVAE_N_SAMPLES = 16

required = [
    "cvae_artifacts",
    "predict_sci_coefficients_cvae",
    "context_cols",
    "build_triplet_coef_dataset",
    "reconstruct_component_spectra",
    "_infer_base_dir_for_reconstruction",
]
missing = [k for k in required if k not in globals()]
if missing:
    raise RuntimeError("Run the cVAE training cells first. Missing: " + ", ".join(missing))

# 1) Load coefficients/context from every10 decomposition products.
e10_triplet = build_triplet_coef_dataset(
    input_fits_path=EVERY10_INPUT,
    sky_near_decomp_fits_path=EVERY10_NEAR,
    sky_far_decomp_fits_path=EVERY10_FAR,
    sci_decomp_fits_path=EVERY10_SCI,
    context_columns=context_cols,
    return_chi2=False,
)
n_e10 = int(e10_triplet["n_rows"])

# 2) Load observed spectra, wavelength grid, and LSF from every10 input.
with fits.open(EVERY10_INPUT) as hdul:
    for ext in ("FLUX_SKY_NEAR", "FLUX_SKY_FAR", "FLUX_SCI", "WAVE", "LSF_SCI"):
        if ext not in hdul:
            raise KeyError(f"Missing extension {ext} in {EVERY10_INPUT}")

    wave_arr = np.asarray(hdul["WAVE"].data, dtype=np.float64)
    flux_near_all = np.asarray(hdul["FLUX_SKY_NEAR"].data, dtype=np.float64)
    flux_far_all = np.asarray(hdul["FLUX_SKY_FAR"].data, dtype=np.float64)
    flux_sci_true_all = np.asarray(hdul["FLUX_SCI"].data, dtype=np.float64)
    lsf_sci_arr = np.asarray(hdul["LSF_SCI"].data, dtype=np.float64)

n_spec, n_wave = flux_sci_true_all.shape
if n_e10 != n_spec:
    raise ValueError(
        f"Row count mismatch: triplet has {n_e10} rows, spectra have {n_spec} rows. "
        "Check that every10 decomposition files were produced from the same input file."
    )

idx_row = int(REQUESTED_ROW)
if idx_row < 0 or idx_row >= n_spec:
    raise IndexError(f"REQUESTED_ROW={idx_row} out of range [0, {n_spec-1}]")

# Normalize WAVE/LSF arrays to per-row vectors, then select requested row.
wave_row = wave_arr if wave_arr.ndim == 1 else wave_arr[idx_row]
lsf_row = lsf_sci_arr if lsf_sci_arr.ndim == 1 else lsf_sci_arr[idx_row]

flux_near_row = flux_near_all[idx_row]
flux_far_row = flux_far_all[idx_row]
flux_sci_true_row = flux_sci_true_all[idx_row]

# 3) Predict SCI coefficients for the requested row only using cVAE.
coef_pred_row_batch = predict_sci_coefficients_cvae(
    cvae_artifacts,
    coef_near_phys=e10_triplet["coef_near"][idx_row:idx_row + 1],
    coef_far_phys=e10_triplet["coef_far"][idx_row:idx_row + 1],
    ctx_near_phys=e10_triplet["ctx_near"][idx_row:idx_row + 1],
    ctx_far_phys=e10_triplet["ctx_far"][idx_row:idx_row + 1],
    ctx_sci_phys=e10_triplet["ctx_sci"][idx_row:idx_row + 1],
    deterministic=bool(CVAE_DETERMINISTIC),
    n_samples=int(CVAE_N_SAMPLES),
    seed=42,
)
coef_pred_row = np.asarray(coef_pred_row_batch[0], dtype=np.float64)

# 4) Reconstruct only this row (fast path).
base_dir_guess = _infer_base_dir_for_reconstruction()
comps_i = reconstruct_component_spectra(
    wave=wave_row,
    coef=coef_pred_row,
    lsf_sigma=lsf_row / 2.35,
    n_spline_knots=25,
    base_dir=base_dir_guess,
)
flux_sci_pred_row = np.asarray(comps_i["total"], dtype=np.float64) / FACTOR

# 5) Single-row metrics.
resid_row = flux_sci_pred_row - flux_sci_true_row
rmse_row = float(np.sqrt(np.mean(resid_row ** 2)))
mae_row = float(np.mean(np.abs(resid_row)))
rel_resid_row = resid_row / np.where(flux_sci_true_row != 0, flux_sci_true_row, np.nan)

mode_txt = "deterministic prior-mean" if CVAE_DETERMINISTIC else f"MC mean ({int(CVAE_N_SAMPLES)} samples)"
print("Single-row reconstruction summary (every10, cVAE coefficients)")
print(f"  row index  = {idx_row}")
print(f"  n_wave     = {n_wave}")
print(f"  cVAE mode  = {mode_txt}")
print(f"  row RMSE   = {rmse_row:.6g}")
print(f"  row MAE    = {mae_row:.6g}")

# 6) Two-panel diagnostic plot: spectra (top, log) + relative residual (bottom).
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.06,
    subplot_titles=("Spectra", "(pred - true) / true"),
    row_heights=[0.7, 0.3],
)

fig.add_trace(go.Scattergl(
    x=wave_row, y=flux_sci_true_row * FACTOR, mode="lines",
    name="science true", line=dict(color="#1f77b4", width=1.2)
), row=1, col=1)
fig.add_trace(go.Scattergl(
    x=wave_row, y=flux_sci_pred_row * FACTOR, mode="lines",
    name=f"pred from cVAE ({mode_txt})", line=dict(color="#d62728", width=1.2)
), row=1, col=1)
fig.add_trace(go.Scattergl(
    x=wave_row, y=flux_near_row * FACTOR, mode="lines",
    name="near obs", line=dict(color="#9467bd", width=0.9, dash="dot")
), row=1, col=1)
fig.add_trace(go.Scattergl(
    x=wave_row, y=flux_far_row * FACTOR, mode="lines",
    name="far obs", line=dict(color="#2ca02c", width=0.9, dash="dot")
), row=1, col=1)

fig.add_trace(go.Scattergl(
    x=wave_row, y=rel_resid_row, mode="lines",
    name="residual recon", line=dict(color="#d62728", width=1.0)
), row=2, col=1)
fig.add_hline(y=0, line=dict(color="black", width=0.8, dash="dash"), row=2, col=1)

fig.update_yaxes(type="log", title_text="Flux", row=1, col=1)
fig.update_yaxes(type="linear", title_text="(pred-true) / true", row=2, col=1)
fig.update_xaxes(title_text="Wavelength [A]", row=2, col=1)

fig.update_layout(
    template="plotly_white",
    title=f"Every10 row {idx_row} | cVAE {mode_txt} | RMSE recon={rmse_row:.4g}",
    height=700,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0.0),
)
fig.show()

Triplet dataset built: n_rows=1726, n_coef=442, n_ctx=6
Single-row reconstruction summary (every10, cVAE coefficients)
  row index  = 1000
  n_wave     = 12401
  cVAE mode  = deterministic prior-mean
  row RMSE   = 6.97411e-14
  row MAE    = 1.35551e-14
